# 08 - Enterprise Bronze to Silver

Conforms, deduplicates, validates, and quarantines the enterprise domain data generated by notebook 07. Silver retains only pseudonymous passenger/workforce keys and deterministic audit metadata.

In [ ]:
from pyspark.sql import functions as F

config = spark.table('bronze_demo_config').first().asDict()
assert int(config['random_seed']) >= 0 and config['is_synthetic'] is True


def conform(target_table, source_table, primary_key, projection, predicate='true'):
    spark.sql(f"""
    CREATE OR REPLACE TABLE {target_table} AS
    WITH ranked AS (
      SELECT *, ROW_NUMBER() OVER (
        PARTITION BY {primary_key}
        ORDER BY ingestion_timestamp DESC, payload_hash DESC
      ) AS _dedupe_rank
      FROM {source_table}
      WHERE is_synthetic = true
    )
    SELECT {projection},
           record_source, source_record_key, batch_id, payload_hash,
           generated_at_utc, generator_version, random_seed,
           'DerivedAnalytical' AS data_classification,
           source_name, source_url, source_as_of_date,
           'Valid' AS data_quality_status,
           CAST(NULL AS STRING) AS rejection_reason,
           true AS is_synthetic
    FROM ranked
    WHERE _dedupe_rank = 1 AND ({predicate})
    """)
    print(target_table, spark.table(target_table).count())


spark.sql("""
CREATE OR REPLACE TABLE silver_quarantine_events AS
SELECT test_case_id, case_type, synthetic_payload, expected_disposition,
       source_event_timestamp, ingestion_timestamp, batch_id, payload_hash,
       generated_at_utc, generator_version, random_seed, record_source,
       'DerivedAnalytical' AS data_classification,
       'IntentionalSyntheticQualityCase' AS rejection_reason,
       true AS is_synthetic
FROM bronze_event_quality_cases
WHERE test_only = true
""")

In [ ]:
# Conformed enterprise dimensions and bridges. Identity-bearing concepts remain pseudonymous tokens only.
conform('dim_organization','bronze_organization','org_unit_id',
        "org_unit_id, parent_org_unit_id, org_unit_type, org_unit_name, operating_region, fictional_relationship_flag",
        "org_unit_id IS NOT NULL AND org_unit_type IN ('CorporateHeadquarters','OperatingRegion')")
conform('dim_route','bronze_route','route_id',
        "route_id, origin_airport_id, destination_airport_id, airline_id, route_category, CAST(distance_km AS INT) AS distance_km, CAST(scheduled_frequency_daily AS INT) AS scheduled_frequency_daily, service_assumption_id",
        "route_id IS NOT NULL AND origin_airport_id <> destination_airport_id AND distance_km > 0")
conform('dim_aircraft_fleet','bronze_aircraft_fleet','aircraft_instance_id',
        "aircraft_instance_id, tail_token, aircraft_type_id, airline_id, base_airport_id, fleet_status, CAST(age_years AS INT) AS age_years, CAST(cumulative_flight_hours_proxy AS BIGINT) AS cumulative_flight_hours_proxy, eligibility_id",
        "aircraft_instance_id IS NOT NULL AND tail_token LIKE 'SYNTH-%'")
conform('dim_work_team','bronze_work_team','work_team_id',
        "work_team_id, airport_id, discipline, team_type, team_label",
        "work_team_id IS NOT NULL AND team_type IN ('MaintenanceTeam','ServiceTeam')")
conform('dim_skill','bronze_skill','skill_id',
        "skill_id, skill_name, skill_category, certification_required", "skill_id IS NOT NULL")
conform('dim_shift','bronze_shift','shift_id',
        "shift_id, shift_name, CAST(start_hour_utc AS INT) AS start_hour_utc, CAST(duration_hours AS INT) AS duration_hours",
        "duration_hours BETWEEN 1 AND 12")
conform('dim_employee','bronze_employee','employee_id',
        "employee_id, home_airport_id, work_team_id, discipline, role, employment_type, training_status, identity_classification",
        "employee_id LIKE 'WRK-%' AND identity_classification = 'PseudonymousSyntheticWorkforceToken'")
conform('bridge_employee_skill','bronze_employee_skill','employee_skill_id',
        "employee_skill_id, employee_id, skill_id, CAST(proficiency_level AS INT) AS proficiency_level",
        "proficiency_level BETWEEN 1 AND 5")
conform('dim_retail_outlet','bronze_retail_outlet','outlet_id',
        "outlet_id, airport_id, terminal_id, outlet_name, outlet_category, concession_model",
        "outlet_id IS NOT NULL AND concession_model = 'SyntheticRevenueShare'")
conform('dim_retail_product','bronze_retail_product','product_id',
        "product_id, product_name, product_category, CAST(unit_price_proxy AS DOUBLE) AS unit_price_proxy",
        "product_id IS NOT NULL AND unit_price_proxy >= 0")
conform('dim_customer','bronze_customer','customer_token',
        "customer_token, passenger_token, customer_segment, profile_classification",
        "customer_token IS NOT NULL AND profile_classification = 'PseudonymousSyntheticProfile'")
conform('dim_passenger','bronze_passenger','passenger_token',
        "passenger_token, customer_segment, CAST(assistance_required AS BOOLEAN) AS assistance_required, identity_classification",
        "passenger_token IS NOT NULL AND identity_classification = 'PseudonymousSyntheticToken'")

spark.sql("""
CREATE OR REPLACE TABLE dim_customer_segment AS
SELECT customer_segment AS customer_segment_id, customer_segment AS customer_segment_name,
       'DerivedAnalytical' AS data_classification, true AS is_synthetic
FROM dim_passenger GROUP BY customer_segment
""")

In [ ]:
# Conformed facts with UTC timestamps, controlled vocabularies, and quality metadata.
conform('bridge_flight_route','bronze_flight_route','flight_event_id',
        "flight_event_id, route_id, destination_airport_id, aircraft_instance_id",
        "flight_event_id IS NOT NULL AND route_id IS NOT NULL AND aircraft_instance_id IS NOT NULL")
conform('fact_flight_leg','bronze_flight_leg','leg_id',
        "leg_id, flight_event_id, route_id, origin_airport_id, destination_airport_id, CAST(scheduled_departure_utc AS TIMESTAMP) AS scheduled_departure_utc, CAST(actual_departure_utc AS TIMESTAMP) AS actual_departure_utc, CAST(scheduled_arrival_utc AS TIMESTAMP) AS scheduled_arrival_utc, CAST(actual_arrival_utc AS TIMESTAMP) AS actual_arrival_utc",
        "actual_departure_utc >= scheduled_departure_utc AND actual_arrival_utc >= scheduled_arrival_utc")
conform('fact_employee_roster','bronze_employee_roster','roster_assignment_id',
        "roster_assignment_id, employee_id, airport_id, work_team_id, assigned_terminal, assigned_gate_id, shift_id, shift_name, CAST(shift_start AS TIMESTAMP) AS shift_start_utc, CAST(shift_end AS TIMESTAMP) AS shift_end_utc, CAST(planned_hours AS DOUBLE) AS planned_hours, CAST(actual_hours AS DOUBLE) AS actual_hours, CAST(overtime_hours AS DOUBLE) AS overtime_hours, CAST(date_format(shift_start, 'yyyyMMdd') AS INT) AS date_key",
        "employee_id IS NOT NULL AND shift_end > shift_start AND planned_hours BETWEEN 1 AND 12 AND actual_hours BETWEEN 1 AND 16")
conform('fact_booking','bronze_booking','booking_id',
        "booking_id, passenger_token, flight_event_id, route_id, customer_segment, fare_class, booking_channel, CAST(source_event_timestamp AS TIMESTAMP) AS booking_timestamp_utc, CAST(booked_days_ahead AS INT) AS booked_days_ahead, CAST(ticket_revenue_proxy AS DOUBLE) AS ticket_revenue_proxy, CAST(checked_bag_count AS INT) AS checked_bag_count, booking_status",
        "booking_id IS NOT NULL AND ticket_revenue_proxy >= 0 AND checked_bag_count BETWEEN 0 AND 2")
conform('fact_boarding_event','bronze_boarding_event','boarding_event_id',
        "boarding_event_id, booking_id, passenger_token, flight_event_id, gate_id, boarding_status, CAST(boarding_timestamp_utc AS TIMESTAMP) AS boarding_timestamp_utc, boarding_window_risk",
        "boarding_event_id IS NOT NULL AND boarding_status IN ('Boarded','NotBoarded')")
conform('fact_baggage_journey','bronze_baggage_journey','bag_token',
        "bag_token, booking_id, flight_event_id, origin_airport_id, destination_airport_id, CAST(loaded_timestamp AS TIMESTAMP) AS loaded_timestamp_utc, CAST(reclaim_timestamp AS TIMESTAMP) AS reclaim_timestamp_utc, journey_status, CAST(mishandled_flag AS BOOLEAN) AS mishandled_flag, CAST(scan_count AS INT) AS expected_scan_count, ROUND((unix_timestamp(reclaim_timestamp) - unix_timestamp(loaded_timestamp)) / 60.0, 2) AS journey_minutes",
        "bag_token IS NOT NULL AND reclaim_timestamp >= loaded_timestamp AND journey_status IN ('Delivered','Exception')")
conform('fact_baggage_scan','bronze_baggage_scan','baggage_scan_id',
        "baggage_scan_id, bag_token, flight_event_id, CAST(scan_sequence AS INT) AS scan_sequence, scan_stage, CAST(scan_timestamp_utc AS TIMESTAMP) AS scan_timestamp_utc",
        "scan_sequence BETWEEN 1 AND 4 AND scan_timestamp_utc IS NOT NULL")
conform('fact_ramp_service_task','bronze_ramp_service_task','ramp_task_id',
        "ramp_task_id, flight_event_id, airport_id, gate_id, task_name, CAST(task_sequence AS INT) AS task_sequence, CAST(planned_start_utc AS TIMESTAMP) AS planned_start_utc, CAST(actual_end_utc AS TIMESTAMP) AS actual_end_utc, task_status, ROUND((unix_timestamp(actual_end_utc)-unix_timestamp(planned_start_utc))/60.0,2) AS task_duration_min",
        "task_sequence BETWEEN 1 AND 5 AND actual_end_utc >= planned_start_utc")
conform('fact_maintenance_work_order','bronze_maintenance_work_order','work_order_id',
        "work_order_id, maintenance_id, airport_id, gate_id, asset_type, team_id, work_order_type, CAST(opened_at_utc AS TIMESTAMP) AS opened_at_utc, CAST(resolved_at_utc AS TIMESTAMP) AS resolved_at_utc, CAST(resolution_hours AS DOUBLE) AS resolution_hours, status, approval_status",
        "resolved_at_utc >= opened_at_utc AND resolution_hours >= 0")
conform('fact_retail_pos','bronze_retail_pos','pos_event_id',
        "pos_event_id, outlet_id, product_id, airport_id, terminal_id, CAST(source_event_timestamp AS TIMESTAMP) AS event_timestamp_utc, CAST(event_hour AS INT) AS event_hour, CAST(transaction_count AS BIGINT) AS transaction_count, CAST(gross_sales_proxy AS DOUBLE) AS gross_sales_proxy, CAST(refund_proxy AS DOUBLE) AS refund_proxy, CAST(average_basket_proxy AS DOUBLE) AS average_basket_proxy, CAST(date_format(source_event_timestamp, 'yyyyMMdd') AS INT) AS date_key",
        "transaction_count >= 0 AND gross_sales_proxy >= refund_proxy")
conform('fact_turnaround_phase','bronze_turnaround_phase','phase_event_id',
        "phase_event_id, flight_event_id, airport_id, gate_id, phase_name, CAST(phase_sequence AS INT) AS phase_sequence, CAST(phase_start AS TIMESTAMP) AS phase_start_utc, CAST(phase_end AS TIMESTAMP) AS phase_end_utc, CAST(phase_duration_min AS DOUBLE) AS phase_duration_min, milestone_status, CAST(date_format(phase_start, 'yyyyMMdd') AS INT) AS date_key",
        "phase_sequence BETWEEN 1 AND 5 AND phase_end >= phase_start")
conform('fact_customer_experience','bronze_customer_experience','cx_event_id',
        "cx_event_id, flight_event_id, route_id, airport_id, customer_segment, CAST(respondent_count AS BIGINT) AS respondent_count, CAST(satisfaction_score AS DOUBLE) AS satisfaction_score, CAST(nps_proxy AS INT) AS nps_proxy, CAST(source_event_timestamp AS TIMESTAMP) AS event_timestamp_utc, CAST(date_format(source_event_timestamp, 'yyyyMMdd') AS INT) AS date_key",
        "satisfaction_score BETWEEN 1 AND 5 AND nps_proxy BETWEEN -100 AND 100")
conform('fact_recommendation','bronze_recommendation_event','recommendation_id',
        "recommendation_id, airport_id, recommendation_type, recommendation_text, recommendation_status, CAST(approval_required AS BOOLEAN) AS approval_required, approved_by_token, CAST(advisory_only AS BOOLEAN) AS advisory_only, CAST(source_event_timestamp AS TIMESTAMP) AS observation_timestamp_utc",
        "advisory_only = true AND approval_required = true")

In [ ]:
# Aircraft rotations, retail inventory, and asset inspections.
conform('fact_aircraft_rotation','bronze_aircraft_rotation','rotation_id',
        "rotation_id, aircraft_instance_id, CAST(rotation_sequence AS INT) AS rotation_sequence, flight_event_id, previous_flight_event_id, CAST(ground_interval_min AS DOUBLE) AS ground_interval_min, CAST(overlap_flag AS BOOLEAN) AS overlap_flag",
        "rotation_sequence >= 1 AND overlap_flag = false AND (ground_interval_min IS NULL OR ground_interval_min >= 0)")
conform('fact_retail_inventory','bronze_retail_inventory','inventory_snapshot_id',
        "inventory_snapshot_id, outlet_id, product_id, airport_id, terminal_id, CAST(on_hand_units AS INT) AS on_hand_units, CAST(reorder_point_units AS INT) AS reorder_point_units, stock_status, CAST(source_event_timestamp AS TIMESTAMP) AS snapshot_timestamp_utc",
        "on_hand_units >= 0 AND reorder_point_units >= 0 AND stock_status IN ('Available','Reorder')")
conform('fact_asset_inspection','bronze_asset_inspection','inspection_id',
        "inspection_id, asset_id, airport_id, gate_id, inspection_type, CAST(inspection_score AS INT) AS inspection_score, inspection_status, CAST(inspected_at_utc AS TIMESTAMP) AS inspected_at_utc, CAST(follow_up_required AS BOOLEAN) AS follow_up_required",
        "inspection_score BETWEEN 0 AND 100 AND inspection_status IN ('Pass','Watch','Action')")

In [ ]:
contracts = {
    'dim_organization':'org_unit_id','dim_route':'route_id','dim_aircraft_fleet':'aircraft_instance_id',
    'dim_work_team':'work_team_id','dim_skill':'skill_id','dim_shift':'shift_id','dim_employee':'employee_id',
    'bridge_employee_skill':'employee_skill_id','dim_retail_outlet':'outlet_id','dim_retail_product':'product_id',
    'dim_customer':'customer_token','dim_passenger':'passenger_token','bridge_flight_route':'flight_event_id',
    'fact_flight_leg':'leg_id','fact_employee_roster':'roster_assignment_id','fact_booking':'booking_id',
    'fact_boarding_event':'boarding_event_id','fact_baggage_journey':'bag_token','fact_baggage_scan':'baggage_scan_id',
    'fact_ramp_service_task':'ramp_task_id','fact_maintenance_work_order':'work_order_id',
    'fact_retail_pos':'pos_event_id','fact_turnaround_phase':'phase_event_id',
    'fact_customer_experience':'cx_event_id','fact_recommendation':'recommendation_id'}
for table_name, primary_key in contracts.items():
    frame = spark.table(table_name)
    assert frame.count() > 0
    assert frame.filter(~F.col('is_synthetic') | (F.col('data_quality_status') != 'Valid')).count() == 0
    assert frame.groupBy(primary_key).count().filter(F.col('count') > 1).count() == 0


def assert_no_orphans(child_table, child_key, parent_table, parent_key):
    child = spark.table(child_table).select(F.col(child_key).alias('key')).where(F.col('key').isNotNull()).distinct()
    parent = spark.table(parent_table).select(F.col(parent_key).alias('key')).where(F.col('key').isNotNull()).distinct()
    orphan_count = child.join(parent, 'key', 'left_anti').count()
    assert orphan_count == 0, child_table + '.' + child_key + ' has ' + str(orphan_count) + ' orphans'


for child_table, child_key, parent_table, parent_key in [
    ('dim_route','origin_airport_id','dim_airport','airport_id'),('dim_route','destination_airport_id','dim_airport','airport_id'),
    ('dim_route','airline_id','dim_airline','airline_id'),('dim_aircraft_fleet','aircraft_type_id','dim_aircraft','aircraft_type_id'),
    ('dim_employee','work_team_id','dim_work_team','work_team_id'),('bridge_employee_skill','employee_id','dim_employee','employee_id'),
    ('bridge_employee_skill','skill_id','dim_skill','skill_id'),('fact_employee_roster','shift_id','dim_shift','shift_id'),
    ('fact_employee_roster','employee_id','dim_employee','employee_id'),('fact_employee_roster','assigned_gate_id','dim_gate','gate_id'),
    ('bridge_flight_route','route_id','dim_route','route_id'),('bridge_flight_route','aircraft_instance_id','dim_aircraft_fleet','aircraft_instance_id'),
    ('fact_flight_leg','flight_event_id','fact_flight_turnaround_events','flight_event_id'),('fact_booking','passenger_token','dim_passenger','passenger_token'),
    ('fact_booking','route_id','dim_route','route_id'),('fact_boarding_event','booking_id','fact_booking','booking_id'),
    ('fact_baggage_journey','booking_id','fact_booking','booking_id'),('fact_baggage_scan','bag_token','fact_baggage_journey','bag_token'),
    ('fact_ramp_service_task','flight_event_id','fact_flight_turnaround_events','flight_event_id'),
    ('fact_maintenance_work_order','maintenance_id','fact_maintenance_events','maintenance_id'),
    ('fact_retail_pos','outlet_id','dim_retail_outlet','outlet_id'),('fact_retail_pos','product_id','dim_retail_product','product_id'),
    ('fact_turnaround_phase','flight_event_id','fact_flight_turnaround_events','flight_event_id'),
    ('fact_customer_experience','route_id','dim_route','route_id'),('fact_recommendation','airport_id','dim_airport','airport_id')]:
    assert_no_orphans(child_table, child_key, parent_table, parent_key)

quarantine = spark.table('silver_quarantine_events')
assert quarantine.count() == 12
assert {row['case_type'] for row in quarantine.select('case_type').distinct().collect()} == {'LateArrival','DuplicateEvent','MalformedPayload','OutOfOrderEvent'}
assert spark.table('fact_booking').count() == spark.table('dim_passenger').count()
assert spark.table('fact_boarding_event').count() == spark.table('fact_booking').count()

compatibility_failures = spark.sql("""
SELECT COUNT(*) AS failures
FROM fact_flight_turnaround_events f
JOIN dim_gate g ON f.gate_id = g.gate_id
JOIN dim_aircraft a ON f.aircraft_type_id = a.aircraft_type_id
WHERE a.wingspan_m > g.max_wingspan_m
""").first()['failures']
assert compatibility_failures == 0
service_failures = spark.sql("""
SELECT COUNT(*) AS failures FROM dim_route r
LEFT ANTI JOIN bronze_airline_airport_service s
ON r.origin_airport_id=s.airport_id AND r.airline_id=s.airline_id
""").first()['failures']
assert service_failures == 0
eligibility_failures = spark.sql("""
SELECT COUNT(*) AS failures FROM dim_aircraft_fleet f
LEFT ANTI JOIN bronze_airline_aircraft_eligibility e
ON f.airline_id=e.airline_id AND f.aircraft_type_id=e.aircraft_type_id
""").first()['failures']
assert eligibility_failures == 0

forbidden_columns = {'name','email','phone','address','passport','biometric','credential','tenant_id','payment'}
for sensitive_table in ['dim_passenger','dim_customer','fact_booking','dim_employee']:
    assert not forbidden_columns.intersection({column.lower() for column in spark.table(sensitive_table).columns})
print('PASS: enterprise Silver quality, quarantine, compatibility, timestamp, pseudonymization, and referential integrity')

In [ ]:
# Additional domain contracts.
for table_name, primary_key in {
    'fact_aircraft_rotation':'rotation_id',
    'fact_retail_inventory':'inventory_snapshot_id',
    'fact_asset_inspection':'inspection_id'}.items():
    frame = spark.table(table_name)
    assert frame.count() > 0
    assert frame.filter(~F.col('is_synthetic') | (F.col('data_quality_status') != 'Valid')).count() == 0
    assert frame.groupBy(primary_key).count().filter(F.col('count') > 1).count() == 0

assert_no_orphans('fact_aircraft_rotation','aircraft_instance_id','dim_aircraft_fleet','aircraft_instance_id')
assert_no_orphans('fact_aircraft_rotation','flight_event_id','fact_flight_turnaround_events','flight_event_id')
assert_no_orphans('fact_retail_inventory','outlet_id','dim_retail_outlet','outlet_id')
assert_no_orphans('fact_retail_inventory','product_id','dim_retail_product','product_id')
assert_no_orphans('fact_asset_inspection','asset_id','dim_asset','asset_id')
assert spark.table('fact_aircraft_rotation').filter(F.col('overlap_flag') | (F.col('ground_interval_min') < 0)).count() == 0
assert spark.table('fact_retail_inventory').filter((F.col('on_hand_units') < 0) | (F.col('reorder_point_units') < 0)).count() == 0
assert spark.table('fact_asset_inspection').filter(~F.col('inspection_score').between(0,100)).count() == 0
print('PASS: aircraft rotation, retail inventory, and asset inspection Silver contracts')